# turkish_distill: Colab pipeline (distillation)

Thin notebook. All logic lives in `src/turkish_distill`. Run the cells top to bottom; every step is resumable, so after a disconnect re-run from the setup cell and then the step you were in.

Two kinds of configs:
- **Stage 1** (`configs/stage1_*.yaml`): single- and two-teacher distillation on the texts of the three QA datasets (`data/stage1` on Drive, made locally by `prepare_pairs`). Scored with the same validation retrieval metric (`val/mrr@10`) as the contrastive runs, so the methods are comparable.
- **Cosmos** (`configs/single_teacher_qwen3_8b.yaml`, `multi_teacher_a.yaml`): the later main training on the large Cosmos corpus.

Before the first run: add `HF_TOKEN` to Colab Secrets (needed for gated models) and set `REPO_URL` below.

## 1. Setup (Drive, repo, install)

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

## 2. Choose the experiment

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# --- Experiment choice ---
# Stage 1 (texts from the three QA datasets, data/stage1 on Drive):
#   configs/stage1_single_qwen3_8b.yaml        one teacher: Qwen3-Embedding-8B
#   configs/stage1_single_bge_m3.yaml          one teacher: BGE-M3 (control)
#   configs/stage1_two_qwen3_8b_bge_m3.yaml    two teachers; set the weights below
# Cosmos corpus (later, main training): configs/single_teacher_qwen3_8b.yaml, configs/multi_teacher_a.yaml
CONFIG = 'configs/stage1_single_qwen3_8b.yaml'
RUN_ID = None   # set e.g. '20260101-120000-abc1234' to resume an existing run; None starts a new run
OVERRIDES = []  # e.g. ['train.max_steps=200'] for a quick trial

# Only for the two-teacher config: the loss weight of each teacher (only the ratio matters).
BGE_WEIGHT, QWEN_WEIGHT = None, None   # e.g. 2.0, 1.0
if BGE_WEIGHT is not None:
    OVERRIDES += [f'teachers.0.weight={BGE_WEIGHT}', f'teachers.1.weight={QWEN_WEIGHT}',
                  f'name=stage1_two_bge{BGE_WEIGHT:g}_qwen{QWEN_WEIGHT:g}']   # the name keeps the runs apart

override_args = ' '.join(f'--override {o}' for o in OVERRIDES)
print(CONFIG, RUN_ID, OVERRIDES)

## 3. Corpus

The corpus path comes from the config. **Stage 1:** `data/stage1/corpus_train.jsonl` must already be on Drive (made locally by `prepare_pairs`, benchmark texts already removed). **Cosmos:** built here once from a Hugging Face dataset (check its text column first); decontamination is not done in that step.

In [ ]:
from turkish_distill.config import load_config

CORPUS = load_config(CONFIG, OVERRIDES).data.corpus_path
if os.path.exists(CORPUS):
    print('corpus ready:', CORPUS)
elif CORPUS.endswith('/data/corpus.jsonl'):
    os.makedirs(os.path.dirname(CORPUS), exist_ok=True)
    !python -m turkish_distill.prepare_corpus --dataset ytu-ce-cosmos/Cosmos-Turkish-Corpus-v1.0 --text-column text --max-samples 1000000 --out {CORPUS}
else:
    raise FileNotFoundError(f'{CORPUS} not found. The stage-1 corpus is made locally by prepare_pairs: upload data/stage1 to Drive.')

## 4. Teacher embeddings (cached on Drive, resumable)

Needs a GPU with enough memory for the largest teacher (Qwen3-8B in bf16 is about 16 GB of weights).

In [ ]:
!python -m turkish_distill.encode_teachers --config {CONFIG} {override_args}

## 5. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/bitirme/runs

## 6. Train (resumes automatically if RUN_ID points to an existing run)

In [ ]:
run_id_arg = f'--run-id {RUN_ID}' if RUN_ID else ''
!python -m turkish_distill.train --config {CONFIG} {run_id_arg} {override_args}

## 7. Inspect metrics

In [ ]:
import glob, pandas as pd
from turkish_distill.logging_utils import read_metrics

paths = sorted(glob.glob(f'{DRIVE_ROOT}/runs/*/*/metrics.jsonl'), key=os.path.getmtime)
print(*paths[-5:], sep='\n')
df = pd.DataFrame(read_metrics(paths[-1]))
cols = [c for c in df.columns if c.startswith(('diag/', 'val/'))]
df[['step'] + cols].dropna(how='all', subset=cols).tail(10)

## 8. Compare the stage-1 runs (contrastive and distillation)

The last validation row of every stage-1 run, best first, on the same held-out pairs. This is the selection signal for the method; TR-MTEB is run separately, once, on the chosen models.

In [ ]:
import glob, yaml, pandas as pd
from turkish_distill.logging_utils import read_metrics

rows = []
for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/*/*')):
    try:
        cfg = yaml.safe_load(open(f'{d}/config.yaml'))
        val = [r for r in read_metrics(f'{d}/metrics.jsonl') if 'val/mrr@10' in r]
    except Exception:
        continue
    if not val:
        continue
    if 'teachers' in cfg:   # distillation run: show the teachers and their weights
        method = 'distill: ' + ' + '.join(f"{t['name']}x{t.get('weight', 1.0):g}" for t in cfg['teachers'])
    else:                   # contrastive run
        method = f"contrastive: {cfg['loss']['name']} s{cfg['loss']['scale']:g} b{cfg['train']['batch_size']}"
    rows.append({'run': d.split('/runs/')[1], 'method': method, 'step': val[-1]['step'],
                 'mrr@10 start': val[0]['val/mrr@10'], 'mrr@10': val[-1]['val/mrr@10'],
                 'recall@1': val[-1]['val/recall@1'], 'recall@10': val[-1]['val/recall@10']})
pd.DataFrame(rows).sort_values('mrr@10', ascending=False).round(4) if rows else print('no finished validation rows yet')